<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/google/meridian/blob/main/demo/Meridian_Getting_Started.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/google/meridian/blob/main/demo/Meridian_Getting_Started.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
  </td>
</table>

# **Introduction to Meridian Demo**

Welcome to the Meridian end-to-end demo. This simplified demo showcases the fundamental functionalities and basic usage of the library, including working examples of the major modeling steps:


<ol start="0">
  <li><a href="#install">Install</a></li>
  <li><a href="#load-data">Load the data</a></li>
  <li><a href="#configure-model">Configure the model</a></li>
  <li><a href="#model-diagnostics">Run model diagnostics</a></li>
  <li><a href="#generate-summary">Generate model results & two-page output</a></li>
  <li><a href="#generate-optimize">Run budget optimization & two-page output</a></li>
  <li><a href="#save-model">Save the model object</a></li>
</ol>


Note that this notebook skips all of the exploratory data analysis and preprocessing steps. It assumes that you have completed these tasks before reaching this point in the demo.

This notebook utilizes sample data. As a result, the numbers and results obtained might not accurately reflect what you encounter when working with a real dataset.

<a name="install"></a>
## Step 0: Install

1\. Make sure you are using one of the available GPU Colab runtimes which is **required** to run Meridian. You can change your notebook's runtime in `Runtime > Change runtime type` in the menu. All users can use the T4 GPU runtime which is sufficient to run the demo colab, free of charge. Users who have purchased one of Colab's paid plans have access to premium GPUs (such as V100, A100 or L4 Nvidia GPU).

2\. Install the latest version of Meridian, and verify that GPU is available.

In [ ]:
# Install meridian: from PyPI @ latest release
!pip install --upgrade google-meridian[colab,and-cuda]

# Install meridian: from PyPI @ specific version
# !pip install google-meridian[colab,and-cuda]==1.1.1

# Install meridian: from GitHub @HEAD
# !pip install --upgrade "google-meridian[colab,and-cuda] @ git+https://github.com/google/meridian.git@main"

In [ ]:
import arviz as az
import IPython
from meridian import constants
from meridian.analysis import analyzer
from meridian.analysis import formatter
from meridian.analysis import optimizer
from meridian.analysis import summarizer
from meridian.analysis import visualizer
from meridian.data import data_frame_input_data_builder
from meridian.data import test_utils
from meridian.model import model
from meridian.model import prior_distribution
from meridian.model import spec
import numpy as np
import pandas as pd
# check if GPU is available
from psutil import virtual_memory
import tensorflow as tf
import tensorflow_probability as tfp

ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))
print(
    'Num GPUs Available: ',
    len(tf.config.experimental.list_physical_devices('GPU')),
)
print(
    'Num CPUs Available: ',
    len(tf.config.experimental.list_physical_devices('CPU')),
)

<a name="load-data"></a>
## Step 1: Load the data

Load the [simulated dataset in CSV format](https://github.com/google/meridian/blob/main/meridian/data/simulated_data/csv/geo_all_channels.csv) as follows.

1\. Read the data into a Pandas DataFrame.

In [ ]:
df = pd.read_csv(
    "https://raw.githubusercontent.com/google/meridian/refs/heads/main/meridian/data/simulated_data/csv/geo_all_channels.csv"
)

2\. Create a DataFrameInputDataBuilder instance.

In [ ]:
builder = data_frame_input_data_builder.DataFrameInputDataBuilder(
    kpi_type='non_revenue',
    default_kpi_column='conversions',
    default_revenue_per_kpi_column='revenue_per_conversion',
)

3\. Offer the components to the builder. Note that the components may be offered all at once or piecewise.

In [ ]:
builder = (
    builder.with_kpi(df)
    .with_revenue_per_kpi(df)
    .with_population(df)
    .with_controls(
        df, control_cols=["sentiment_score_control", "competitor_sales_control"]
    )
)

channels = ["Channel0", "Channel1", "Channel2", "Channel3", "Channel4"]
builder = builder.with_media(
    df,
    media_cols=[f"{channel}_impression" for channel in channels],
    media_spend_cols=[f"{channel}_spend" for channel in channels],
    media_channels=channels,
)

4. If your data includes organic media or non-media treatments, you can add them using `with_organic_media` and `with_non_media_treatments` methods. For the definition of each variable, see
[Collect and organize your data](https://developers.google.com/meridian/docs/user-guide/collect-data)

In [ ]:
builder = builder.with_non_media_treatments(
    df, non_media_treatment_cols=['Promo']
).with_organic_media(
    df,
    organic_media_cols=['Organic_channel0_impression'],
    organic_media_channels=['Organic_channel0'],
)

5. Finally, build the InputData.

In [ ]:
data = builder.build()

Note that the simulated data here does not contain reach and frequency. We recommend including reach and frequency data whenever they are available. For information about the advantages of utilizing reach and frequency, see [Bayesian Hierarchical Media Mix Model Incorporating Reach and Frequency Data](https://research.google/pubs/bayesian-hierarchical-media-mix-model-incorporating-reach-and-frequency-data/#:~:text=By%20incorporating%20R%26F%20into%20MMM,based%20on%20optimal%20frequency%20recommendations.). For code snippet for loading reach and frequency data, see [Load geo-level data with reach and frequency](https://developers.google.com/meridian/docs/user-guide/load-geo-data-with-rf)

The documentation provides guidance for instances where reach and frequency data is accessible for specific channels. Additionally, for information about how to load other data types and formats, including data with reach and frequency, see [Supported data types and formats](https://developers.google.com/meridian/docs/user-guide/supported-data-types-formats).

<a name="configure-model"></a>
## Step 2: Configure the model

Meridian uses Bayesian framework and Markov Chain Monte Carlo (MCMC) algorithms to sample from the posterior distribution.

1\. Inititalize the `Meridian` class by passing the loaded data and the customized model specification. One advantage of Meridian lies in its capacity to calibrate the model directly through ROI priors, as described in [Media Mix Model Calibration With Bayesian Priors](https://research.google/pubs/media-mix-model-calibration-with-bayesian-priors/). In this particular example, the ROI priors for all media channels are identical, with each being represented as Lognormal(0.2, 0.9).

In [ ]:
roi_mu = 0.2  # Mu for ROI prior for each media channel.
roi_sigma = 0.9  # Sigma for ROI prior for each media channel.
prior = prior_distribution.PriorDistribution(
    roi_m=tfp.distributions.LogNormal(roi_mu, roi_sigma, name=constants.ROI_M)
)
model_spec = spec.ModelSpec(prior=prior)

mmm = model.Meridian(input_data=data, model_spec=model_spec)

2\. Use the `sample_prior()` and `sample_posterior()` methods to obtain samples from the prior and posterior distributions of model parameters. If you are using the T4 GPU runtime this step may take about 10 minutes for the provided data set.

In [ ]:
%%time
mmm.sample_prior(500)
mmm.sample_posterior(
    n_chains=10, n_adapt=2000, n_burnin=500, n_keep=1000, seed=0
)

For more information about configuring the parameters and using a customized

---

model specification, such as setting different ROI priors for each media channel, see [Configure the model](https://developers.google.com/meridian/docs/user-guide/configure-model).

<a name="model-diagnostics"></a>
## Step 3: Run model diagnostics

After the model is built, you must assess convergence, debug the model if needed, and then assess the model fit.

1\. Assess convergence. Run the following code to generate r-hat statistics. R-hat close to 1.0 indicate convergence. R-hat < 1.2 indicates approximate convergence and is a reasonable threshold for many problems.

In [ ]:
model_diagnostics = visualizer.ModelDiagnostics(mmm)
model_diagnostics.plot_rhat_boxplot()

2\. Assess the model's fit by comparing the expected sales against the actual sales.

In [ ]:
model_fit = visualizer.ModelFit(mmm)
model_fit.plot_model_fit()

For more information and additional model diagnostics checks, see [Modeling diagnostics](https://developers.google.com/meridian/docs/user-guide/model-diagnostics).

<a name="generate-summary"></a>
## Step 4: Generate model results & two-page output

To export the two-page HTML summary output, initialize the `Summarizer` class with the model object. Then pass in the filename, filepath, start date, and end date to `output_model_results_summary` to run the summary for that time duration and save it to the specified file.

In [ ]:
mmm_summarizer = summarizer.Summarizer(mmm)

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
filepath = '/content/drive/MyDrive'
start_date = '2021-01-25'
end_date = '2024-01-15'
mmm_summarizer.output_model_results_summary(
    'summary_output.html', filepath, start_date, end_date
)

Here is a preview of the two-page output based on the simulated data:

In [ ]:
IPython.display.HTML(filename='/content/drive/MyDrive/summary_output.html')

For a customized two-page report, model results summary table, and individual visualizations, see [Model results report](https://developers.google.com/meridian/docs/user-guide/generate-model-results-report) and [plot media visualizations](https://developers.google.com/meridian/docs/user-guide/plot-media-visualizations).





<a name="generate-optimize"></a>

> Adicionar aspas


## Step 5: Run budget optimization & generate an optimization report

You can choose what scenario to run for the budget allocation. In default scenario, you find the optimal allocation across channels for a given budget to maximize the return on investment (ROI).

1\. Instantiate the `BudgetOptimizer` class and run the `optimize()` method without any customization, to run the default library's Fixed Budget Scenario to maximize ROI.

In [ ]:
%%time
budget_optimizer = optimizer.BudgetOptimizer(mmm)
optimization_results = budget_optimizer.optimize()

2\. Export the 2-page HTML optimization report, which contains optimized spend allocations and ROI.

In [ ]:
filepath = '/content/drive/MyDrive'
optimization_results.output_optimization_summary(
    'optimization_output.html', filepath
)

In [ ]:
IPython.display.HTML(filename='/content/drive/MyDrive/optimization_output.html')

For information about customized optimization scenarios, such as flexible budget scenarios, see [Budget optimization scenarios](https://developers.google.com/meridian/docs/user-guide/budget-optimization-scenarios). For more information about optimization results summary and individual visualizations, see [optimization results output](https://developers.google.com/meridian/docs/user-guide/generate-optimization-results-output) and [optimization visualizations](https://developers.google.com/meridian/docs/user-guide/plot-optimization-visualizations).

<a name="save-model"></a>
## Step 6: Save the model object

We recommend that you save the model object for future use. This helps you to  avoid repetitive model runs and saves time and computational resources. After the model object is saved, you can load it at a later stage to continue the analysis or visualizations without having to re-run the model.


Run the following codes to save the model object:

In [ ]:
file_path = '/content/drive/MyDrive/saved_mmm.pkl'
model.save_mmm(mmm, file_path)

Run the following codes to load the saved model:

In [ ]:
mmm = model.load_mmm(file_path)

In [ ]:
# # Etapa 0: Instale a biblioteca Meridian e as dependências necessárias
# # Certifique-se de que a GPU está ativada no seu ambiente de execução.

# !pip install --upgrade google-meridian[colab, and-cuda]
# !pip install tensorflow-probability==0.23.0
# !pip install tensorflow==2.15.0

import pandas as pd
import numpy as np
import meridian as md
from meridian.data import data_frame_input_data_builder
from meridian.model import model, prior_distribution, spec
from meridian.analysis import visualizer, summarizer, optimizer
import tensorflow as tf
import tensorflow_probability as tfp
import arviz as az
import matplotlib.pyplot as plt
from google.colab import drive
import IPython

# 1. Simular a base de dados com o novo evento "convenção de vendas"
n_days = 100
dates = pd.date_range(start='2025-01-01', periods=n_days, freq='D')
data = {
    'time': dates,
    'google_ads_impressoes': np.random.randint(20000, 40000, size=n_days),
    'google_ads_custo': np.random.randint(400, 700, size=n_days),
    'meta_ads_impressoes': np.random.randint(30000, 45000, size=n_days),
    'meta_ads_custo': np.random.randint(380, 550, size=n_days),
    'linkedin_impressoes': np.random.randint(8000, 11000, size=n_days),
    'linkedin_custo': np.random.randint(150, 220, size=n_days),
    'organic_busca_impressoes': np.random.randint(14000, 18000, size=n_days),
    'feriado': np.random.choice([0,1], size=n_days, p=[0.9,0.1]),
    'atividade_outbound': np.random.randint(100, 180, size=n_days),
    'convencao_vendas': np.random.choice([0,1], size=n_days, p=[0.95,0.05]),
    'conquistados': np.random.randint(3, 8, size=n_days),
}
df = pd.DataFrame(data)
df['time'] = pd.to_datetime(df['time'])

# 2. Preparar e carregar dados
paid_channels = ['google_ads', 'meta_ads', 'linkedin']
paid_media_cols = [f'{c}_impressoes' for c in paid_channels]
paid_spend_cols = [f'{c}_custo' for c in paid_channels]

organic_channels = ['organic_busca']
organic_media_cols = [f'{c}_impressoes' for c in organic_channels]

# Adicionamos 'convencao_vendas' à lista de variáveis não-mídia
non_media_treatments = ['feriado', 'atividade_outbound', 'convencao_vendas']

builder = data_frame_input_data_builder.DataFrameInputDataBuilder(
    kpi_type='non_revenue',
    default_kpi_column='conquistados'
)

# O método with_non_media_treatments lê dados de tratamentos não-mídia de um DataFrame [cite: 1101, 1102]
builder = (
    builder.with_kpi(df)
    .with_media(df, media_cols=paid_media_cols, media_spend_cols=paid_spend_cols, media_channels=paid_channels)
    .with_organic_media(df, organic_media_cols=organic_media_cols, organic_media_channels=organic_channels)
    .with_non_media_treatments(df, non_media_treatment_cols=non_media_treatments)
)
input_data = builder.build()


# 3. Configurar priors de ROI específicos para cada canal
# google_ads: mu de 0.5 e sigma de 0.2. Isso indica que você acredita que o Google Ads terá o maior ROI, e a baixa incerteza (sigma de 0.2) mostra que você tem um alto grau de confiança nessa estimativa.
# meta_ads: mu de 0.3 e sigma de 0.3. Você espera um ROI um pouco menor do que o do Google Ads e tem um nível de incerteza moderado sobre essa estimativa.
# linkedin: mu de 0.1 e sigma de 0.5. Isso sugere que você espera que o LinkedIn tenha o ROI mais baixo de todos os canais pagos, e a incerteza é a maior (sigma de 0.5), indicando menor confiança nessa estimativa.

roi_priors = {
    'google_ads': {'mu': 0.5, 'sigma': 0.2},
    'meta_ads': {'mu': 0.3, 'sigma': 0.3},
    'linkedin': {'mu': 0.1, 'sigma': 0.5},
}
prior = prior_distribution.PriorDistribution(
    roi_m=tfp.distributions.LogNormal(
        loc=[roi_priors[c]['mu'] for c in paid_channels],
        scale=[roi_priors[c]['sigma'] for c in paid_channels]
    )
)
model_spec = spec.ModelSpec(prior=prior)
mmm = model.Meridian(input_data=input_data, model_spec=model_spec)

# 4. Treinar o modelo
print("Amostrando a distribuição de prior...")
mmm.sample_prior(500)
print("Amostrando a distribuição posterior...")
mmm.sample_posterior(n_keep=1000, n_burnin=500, n_adapt=200, n_chains=4)

# 5. Avaliar o modelo com diagnósticos
print("Executando diagnósticos do modelo...")
model_diagnostics = visualizer.ModelDiagnostics(mmm)
model_diagnostics.plot_rhat_boxplot()
plt.show()

# Avaliar o ajuste do modelo
model_fit = visualizer.ModelFit(mmm)
model_fit.plot_model_fit()
plt.show()

# 6. Gerar relatório de resultados
print("Gerando e exportando o relatório de resultados do modelo...")
try:
    drive.mount('/content/drive')
    file_path = '/content/drive/MyDrive'
except Exception as e:
    print(f"Não foi possível montar o Google Drive. Salvando no diretório local. Erro: {e}")
    file_path = '.'

mmm_summarizer = summarizer.Summarizer(mmm)
start_date = '2025-11-01'
end_date = '2025-11-10'
mmm_summarizer.output_model_results_summary(
    'summary_output.html', file_path, start_date, end_date
)
print(f"Relatório salvo em {file_path}/summary_output.html")

# 7. Executar otimização de orçamento (exemplo)
print("Executando otimização de orçamento...")
budget_optimizer = optimizer.BudgetOptimizer(mmm)
optimization_results = budget_optimizer.optimize()
optimization_results.output_optimization_summary(
    'optimization_output.html', file_path
)
print(f"Relatório de otimização salvo em {file_path}/optimization_output.html")

# 8. Plotar as contribuições dos canais
print("Plotando a contribuição dos canais...")
model_summarizer = summarizer.Summarizer(mmm)
model_summarizer.plot_contributions()
plt.show()

# 9. Salvar o objeto do modelo para uso futuro
model.save_mmm(mmm, f'{file_path}/saved_mmm.pkl')
print(f"Modelo salvo em {file_path}/saved_mmm.pkl")

# 10. Exibir os relatórios HTML
print("\nExibindo o relatório de resumo do modelo:")
model_summary_path = f"{file_path}/summary_output.html"
display(IPython.display.HTML(filename=model_summary_path))

print("\nExibindo o relatório de otimização de orçamento:")
optimization_summary_path = f"{file_path}/optimization_output.html"
display(IPython.display.HTML(filename=optimization_summary_path))

In [ ]:
# Etapa 0: Instale a biblioteca Meridian e as dependências compatíveis.
# A versão 1.1.6 do Meridian requer o TensorFlow Probability na versão 0.25 ou superior.
# Vamos instalar uma versão mais recente e compatível do TensorFlow e do TensorFlow Probability.

!pip install --upgrade "google-meridian[colab, and-cuda]"
!pip install tensorflow-probability==0.25.0
!pip install tensorflow==2.16.1

import pandas as pd
import numpy as np
import meridian as md
from meridian.data import data_frame_input_data_builder
from meridian.model import model, prior_distribution, spec
from meridian.analysis import visualizer, summarizer, optimizer
import tensorflow as tf
import tensorflow_probability as tfp
import arviz as az
import matplotlib.pyplot as plt
from google.colab import drive
from IPython.display import display, HTML

# 1. Simular a base de dados com o novo evento "convenção de vendas"
n_days = 100
dates = pd.date_range(start='2025-01-01', periods=n_days, freq='D')
np.random.seed(0) # Para garantir a reprodutibilidade dos dados

data = {
    'time': dates,
    'google_ads_impressoes': np.random.randint(20000, 40000, size=n_days),
    'google_ads_custo': np.random.randint(400, 700, size=n_days),
    'meta_ads_impressoes': np.random.randint(30000, 45000, size=n_days),
    'meta_ads_custo': np.random.randint(380, 550, size=n_days),
    'linkedin_impressoes': np.random.randint(8000, 11000, size=n_days),
    'linkedin_custo': np.random.randint(150, 220, size=n_days),
    'organic_busca_impressoes': np.random.randint(14000, 18000, size=n_days),
    'feriado': np.random.choice([0,1], size=n_days, p=[0.9,0.1]),
    'atividade_outbound': np.random.randint(100, 180, size=n_days),
    'convencao_vendas': np.zeros(n_days, dtype=int),
    'conquistados': np.random.randint(3, 8, size=n_days),
}

# Adicionando a correlação entre a convenção de vendas e o KPI
data['convencao_vendas'][50:55] = 1  # Simula um evento de 5 dias
data['conquistados'][50:55] = np.random.randint(8, 15, 5) # Aumenta o KPI durante o evento

df = pd.DataFrame(data)
df['time'] = pd.to_datetime(df['time'])

# 2. Preparar e carregar dados
paid_channels = ['google_ads', 'meta_ads', 'linkedin']
paid_media_cols = [f'{c}_impressoes' for c in paid_channels]
paid_spend_cols = [f'{c}_custo' for c in paid_channels]

organic_channels = ['organic_busca']
organic_media_cols = [f'{c}_impressoes' for c in organic_channels]

# Adicionamos 'convencao_vendas' à lista de variáveis não-mídia
non_media_treatments = ['feriado', 'atividade_outbound', 'convencao_vendas']

builder = data_frame_input_data_builder.DataFrameInputDataBuilder(
    kpi_type='non_revenue',
    default_kpi_column='conquistados'
)

builder = (
    builder.with_kpi(df)
    .with_media(df, media_cols=paid_media_cols, media_spend_cols=paid_spend_cols, media_channels=paid_channels)
    .with_organic_media(df, organic_media_cols=organic_media_cols, organic_media_channels=organic_channels)
    .with_non_media_treatments(df, non_media_treatment_cols=non_media_treatments)
)
input_data = builder.build()


# 3. Configurar priors de ROI específicos para cada canal
roi_priors = {
    'google_ads': {'mu': 0.5, 'sigma': 0.2},
    'meta_ads': {'mu': 0.3, 'sigma': 0.3},
    'linkedin': {'mu': 0.1, 'sigma': 0.5},
}
prior = prior_distribution.PriorDistribution(
    roi_m=tfp.distributions.LogNormal(
        loc=[roi_priors[c]['mu'] for c in paid_channels],
        scale=[roi_priors[c]['sigma'] for c in paid_channels]
    )
)
model_spec = spec.ModelSpec(prior=prior)
mmm = model.Meridian(input_data=input_data, model_spec=model_spec)

# 4. Treinar o modelo
print("Amostrando a distribuição de prior...")
mmm.sample_prior(500)
print("Amostrando a distribuição posterior...")
# Ajustando os hiperparâmetros para melhor convergência com 100 dias de dados
mmm.sample_posterior(n_keep=1000, n_burnin=500, n_adapt=500, n_chains=4)

# 5. Avaliar o modelo com diagnósticos
print("Executando diagnósticos do modelo...")
model_diagnostics = visualizer.ModelDiagnostics(mmm)
model_diagnostics.plot_rhat_boxplot()
plt.show()

# Avaliar o ajuste do modelo
model_fit = visualizer.ModelFit(mmm)
model_fit.plot_model_fit()
plt.show()

# 6. Gerar relatório de resultados
print("Gerando e exportando o relatório de resultados do modelo...")
try:
    drive.mount('/content/drive')
    file_path = '/content/drive/MyDrive'
except Exception as e:
    print(f"Não foi possível montar o Google Drive. Salvando no diretório local. Erro: {e}")
    file_path = '.'

# Ajustando as datas do relatório para corresponder ao novo período de 100 dias
start_date = '2025-01-01'
end_date = '2025-04-10'

mmm_summarizer = summarizer.Summarizer(mmm)
mmm_summarizer.output_model_results_summary(
    'summary_output.html', file_path, start_date, end_date
)
print(f"Relatório salvo em {file_path}/summary_output.html")

# 7. Executar otimização de orçamento (exemplo)
print("Executando otimização de orçamento...")
budget_optimizer = optimizer.BudgetOptimizer(mmm)
optimization_results = budget_optimizer.optimize()
optimization_results.output_optimization_summary(
    'optimization_output.html', file_path
)
print(f"Relatório de otimização salvo em {file_path}/optimization_output.html")

# 8. Plotar as contribuições dos canais
print("Plotando a contribuição dos canais...")
model_summarizer = summarizer.Summarizer(mmm)
model_summarizer.plot_contributions()
plt.show()

# 9. Salvar o objeto do modelo para uso futuro
model.save_mmm(mmm, f'{file_path}/saved_mmm.pkl')
print(f"Modelo salvo em {file_path}/saved_mmm.pkl")

# 10. Exibir os relatórios HTML
print("\nExibindo o relatório de resumo do modelo:")
model_summary_path = f"{file_path}/summary_output.html"
display(HTML(filename=model_summary_path))

print("\nExibindo o relatório de otimização de orçamento:")
optimization_summary_path = f"{file_path}/optimization_output.html"
display(HTML(filename=optimization_summary_path))